In [1]:
import pandas as pd

In [ ]:
# load synthetic units/households
units = pd.read_csv('output/popsim_hh.csv')
# select only occupied units for the households table
hh = units.loc[units['is_vacant'] == 0].drop(columns=['is_vacant','vacant_type','rent','home_value']).copy()
hh.to_csv('output/synthetic_households.csv', index=False)

# create masks for single-family, multi-family, owner-occupied, and renter-occupied units
sf_mask = units['building_type'].between(1, 4)
mf_mask = units['building_type'].between(4, 10)
owner_mask = (units['tenure'] == 1) | (units['vacant_type'].isin([3,4,5]))
renter_mask = (units['tenure'] == 2) | (units['vacant_type'].isin([1,2,6,7]))

# assign unit type IDs based on the masks
units.loc[sf_mask & owner_mask,'unit_type_id'] = 1
units.loc[mf_mask & owner_mask,'unit_type_id'] = 2
units.loc[sf_mask & renter_mask,'unit_type_id'] = 3
units.loc[mf_mask & renter_mask,'unit_type_id'] = 4

In [ ]:
# calculate average rents and home values by block and save to CSV
rents_values = units.groupby('block_id')[['rent','home_value']].mean().round(2)
rents_values.to_csv('output/synthetic_rents_values.csv')

In [7]:
# assign unique unit IDs and save the housing units table
units['unit_id'] = range(1, len(units) + 1)
units[['unit_id','block_id','year_built','unit_type_id']].to_csv('output/synthetic_housing_units.csv', index=False)

# generate a lookup table for unit types
unit_types = {
    1: 'Single-family (1-4 units) owner-occupied',
    2: 'Multi-family (5+ units) owner-occupied',
    3: 'Single-family (1-4 units) renter-occupied',
    4: 'Multi-family (5+ units) renter-occupied'
}
unit_types_df = pd.DataFrame(list(unit_types.items()), columns=['unit_type_id', 'description'])
unit_types_df.to_csv('output/housing_unit_types.csv', index=False)